In [1]:
py_path <- Sys.which("python")

if (py_path == "") {
  py_path <- Sys.which("python3")
}

Sys.setenv(RETICULATE_PYTHON = py_path)

suppressPackageStartupMessages({
  library(Seurat)
  library(Matrix)
  library(data.table)
  library(reticulate)
  library(CellEnergy)
})

cat("[INFO] Python used by reticulate:\n")
print(reticulate::py_config())

cat("[INFO] Python module check:\n")
cat("pandas:", reticulate::py_module_available("pandas"), "\n")
cat("numpy :", reticulate::py_module_available("numpy"), "\n")
cat("scipy :", reticulate::py_module_available("scipy"), "\n\n")

Warning message:
“no DISPLAY variable so Tk is not available”


[INFO] Python used by reticulate:
python:         /home/zhanghang/Anaconda/envs/r_cellenergy/bin/python
libpython:      /home/zhanghang/Anaconda/envs/r_cellenergy/lib/libpython3.14.so
pythonhome:     /home/zhanghang/Anaconda/envs/r_cellenergy:/home/zhanghang/Anaconda/envs/r_cellenergy
version:        3.14.5 | packaged by conda-forge | (main, May 20 2026, 00:15:36) [GCC 14.3.0]
numpy:          /home/zhanghang/Anaconda/envs/r_cellenergy/lib/python3.14/site-packages/numpy
numpy_version:  2.4.6

NOTE: Python version was forced by RETICULATE_PYTHON
[INFO] Python module check:
pandas: TRUE 
numpy : TRUE 
scipy : TRUE 



In [ ]:
# =========================================================
# GSE158490 / GSE158493 HSPC preprocessing using Barcode.csv
#
# Input:
#   /home/zhanghang/GSE158490/GSE158490_matrix.mtx
#   /home/zhanghang/GSE158490/GSE158490_genes.tsv
#   /home/zhanghang/GSE158490/GSE158490_barcodes.tsv
#   /home/zhanghang/GSE158490/Barcode.csv
#
# Barcode.csv format:
#   cell                  celltype
#   AAACCTGAGAGTAATC-1    GMP
#   AAACCTGAGCGGCTTC-1    CLP
#
# Workflow:
#   1. Read raw expression matrix
#   2. Read Barcode.csv celltype annotation
#   3. Keep only annotated cells
#   4. NormalizeData(scale.factor = 10000)
#   5. Select 2000 HVGs
#   6. Export normalized HVG expression without MinMax
#   7. Calculate GLNE using CellEnergy
#   8. Apply MinMax after GLNE calculation
#   9. Export final View1, View2 and labels
# =========================================================


# =========================================================
# 0. Load packages
# =========================================================

required_pkgs <- c(
  "Seurat",
  "Matrix",
  "data.table",
  "CellEnergy"
)

for (pkg in required_pkgs) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    stop(paste0("[ERROR] Package not installed: ", pkg))
  }
}

suppressPackageStartupMessages({
  library(Seurat)
  library(Matrix)
  library(data.table)
  library(CellEnergy)
})


# =========================================================
# 1. Paths and parameters
# =========================================================

DATA_DIR <- "path/"

MTX_PATH <- file.path(DATA_DIR, "GSE158490_matrix.mtx")
GENE_PATH <- file.path(DATA_DIR, "GSE158490_genes.tsv")
BARCODE_PATH <- file.path(DATA_DIR, "GSE158490_barcodes.tsv")


ANNOTATION_PATH <- file.path(DATA_DIR, "Barcode.csv")

OUT_DIR <- file.path(DATA_DIR, "GSE158490_Barcode_model_ready")
dir.create(OUT_DIR, showWarnings = FALSE, recursive = TRUE)

SCALE_FACTOR <- 10000
N_HVG <- 2000

EXPECTED_CELL_NUMBER <- 5183

OUT_SEURAT_RDS <- file.path(
  OUT_DIR,
  "GSE158490_Barcode_filtered_seurat.rds"
)

OUT_EXPR_NORM_NO_MINMAX <- file.path(
  OUT_DIR,
  "GSE158490_norm_HVG_log1p_no_MinMax_cell_by_gene.csv"
)

OUT_VIEW1_MINMAX <- file.path(
  OUT_DIR,
  "final_norm_HVG_log1p_MinMax_cell_by_gene.csv"
)

OUT_VIEW1_COMPAT <- file.path(
  OUT_DIR,
  "final_raw_HVG_counts_MinMax_cell_by_gene.csv"
)

OUT_GLNE_RAW <- file.path(
  OUT_DIR,
  "GSE158490_GLNE_raw_cell_by_gene.csv"
)

OUT_VIEW2_MINMAX <- file.path(
  OUT_DIR,
  "final_GLNE_MinMax_cell_by_gene.csv"
)

OUT_LABEL <- file.path(
  OUT_DIR,
  "final_cell_labels.csv"
)

OUT_LABEL_COUNTS <- file.path(
  OUT_DIR,
  "label_counts.csv"
)

OUT_HVG <- file.path(
  OUT_DIR,
  "selected_HVGs.txt"
)

OUT_QC <- file.path(
  OUT_DIR,
  "qc_metrics_annotated_cells.csv"
)

OUT_CELLENERGY_RDS <- file.path(
  OUT_DIR,
  "CellEnergy_calcGEn_result.rds"
)

OUT_REPORT <- file.path(
  OUT_DIR,
  "preprocessing_report.txt"
)


# =========================================================
# 2. Helper functions
# =========================================================

make_unique <- function(x) {
  make.unique(as.character(x), sep = "_")
}


normalize_barcode <- function(x) {
  x <- as.character(x)
  x <- trimws(x)
  x <- gsub("\\.", "-", x)
  return(x)
}


standardize_celltype <- function(x) {
  x <- as.character(x)
  x <- trimws(x)

  out <- x

  out[grepl("HSC|MPP|stem|multipotent", x, ignore.case = TRUE)] <- "HSC/MPP"
  out[grepl("^CLP$|lymphoid", x, ignore.case = TRUE)] <- "CLP"
  out[grepl("^GMP$|granulocyte|monocyte", x, ignore.case = TRUE)] <- "GMP"
  out[grepl("^MEP$|megakaryocyte|erythroid", x, ignore.case = TRUE)] <- "MEP"
  out[grepl("^mid$|\\bmid\\b|intermediate", x, ignore.case = TRUE)] <- "mid"

  return(out)
}


read_annotation_file <- function(annotation_path) {
  ann <- fread(
    annotation_path,
    header = TRUE,
    data.table = FALSE,
    check.names = FALSE
  )

  colnames(ann) <- trimws(colnames(ann))

  cat("[INFO] Annotation file dimension:\n")
  print(dim(ann))

  cat("[INFO] Annotation columns:\n")
  print(colnames(ann))

  possible_cell_cols <- c(
    "cell",
    "Cell",
    "barcode",
    "Barcode",
    "BARCODE",
    "cell_barcode",
    "CellBarcode"
  )

  cell_col <- intersect(possible_cell_cols, colnames(ann))

  if (length(cell_col) > 0) {
    cell_col <- cell_col[1]
  } else {
    cell_col <- colnames(ann)[1]
    cat("[WARNING] Cell column not explicitly found. Use first column as cell column:\n")
    cat(cell_col, "\n")
  }

  possible_label_cols <- c(
    "celltype",
    "Celltype",
    "cell_type",
    "Cell_type",
    "CellType",
    "type",
    "Type",
    "label",
    "Label",
    "annotation",
    "Annotation"
  )

  label_col <- intersect(possible_label_cols, colnames(ann))

  if (length(label_col) > 0) {
    label_col <- label_col[1]
  } else {
    if (ncol(ann) < 2) {
      stop("[ERROR] Annotation file must contain at least two columns: cell and celltype.")
    }
    label_col <- colnames(ann)[2]
    cat("[WARNING] Celltype column not explicitly found. Use second column as label column:\n")
    cat(label_col, "\n")
  }

  ann_df <- data.frame(
    Cell = normalize_barcode(ann[[cell_col]]),
    Label = standardize_celltype(ann[[label_col]]),
    stringsAsFactors = FALSE,
    check.names = FALSE
  )

  ann_df <- ann_df[
    !is.na(ann_df$Cell) &
      ann_df$Cell != "" &
      !is.na(ann_df$Label) &
      ann_df$Label != "",
    ,
    drop = FALSE
  ]

  ann_df <- ann_df[!duplicated(ann_df$Cell), , drop = FALSE]

  rownames(ann_df) <- NULL

  return(ann_df)
}


get_assay_data_v5 <- function(obj, assay = "RNA", layer = "data") {
  DefaultAssay(obj) <- assay

  available_layers <- Layers(obj[[assay]])

  cat("[INFO] Assay:", assay, "\n")
  cat("[INFO] Available layers:\n")
  print(available_layers)

  if (!(layer %in% available_layers)) {
    stop(
      paste0(
        "[ERROR] Layer '", layer, "' not found in assay '", assay, "'.\n",
        "Available layers: ",
        paste(available_layers, collapse = ", ")
      )
    )
  }

  x <- GetAssayData(
    object = obj,
    assay = assay,
    layer = layer
  )

  return(x)
}


minmax_scale_by_column <- function(mat) {
  mat <- as.matrix(mat)
  storage.mode(mat) <- "numeric"

  col_min <- apply(mat, 2, min, na.rm = TRUE)
  col_max <- apply(mat, 2, max, na.rm = TRUE)

  denom <- col_max - col_min
  denom[denom == 0 | is.na(denom)] <- 1

  mat_scaled <- sweep(mat, 2, col_min, FUN = "-")
  mat_scaled <- sweep(mat_scaled, 2, denom, FUN = "/")

  mat_scaled[is.na(mat_scaled)] <- 0

  return(mat_scaled)
}


write_matrix_csv <- function(mat, file_path) {
  df <- as.data.frame(mat, check.names = FALSE)
  write.csv(df, file_path, quote = FALSE)
}


run_calcGEn_compat <- function(input_file) {
  result <- tryCatch(
    {
      calcGEn(input_file, verbose = TRUE)
    },
    error = function(e1) {
      cat("[WARNING] calcGEn(file, verbose = TRUE) failed. Try calcGEn(file)...\n")
      calcGEn(input_file)
    }
  )

  return(result)
}


find_glne_from_result <- function(result_obj) {
  if (is.matrix(result_obj) || is.data.frame(result_obj) || inherits(result_obj, "Matrix")) {
    return(result_obj)
  }

  if (!is.list(result_obj)) {
    stop("[ERROR] calcGEn result is not a list/matrix/data.frame.")
  }

  possible_names <- c(
    "GLNE",
    "glne",
    "GEn",
    "GEN",
    "geneLocalNetworkEnergy",
    "GeneLocalNetworkEnergy"
  )

  for (nm in possible_names) {
    if (nm %in% names(result_obj)) {
      obj <- result_obj[[nm]]
      if (is.matrix(obj) || is.data.frame(obj) || inherits(obj, "Matrix")) {
        return(obj)
      }
    }
  }

  for (nm in names(result_obj)) {
    obj <- result_obj[[nm]]
    if (is.matrix(obj) || is.data.frame(obj) || inherits(obj, "Matrix")) {
      if (nrow(obj) > 10 && ncol(obj) > 10) {
        cat("[WARNING] GLNE name not found. Use matrix-like object:", nm, "\n")
        return(obj)
      }
    }
  }

  stop("[ERROR] Could not find GLNE matrix from calcGEn result.")
}


fix_glne_orientation <- function(glne_obj, expected_cells, expected_genes) {
  glne_df <- as.data.frame(glne_obj, check.names = FALSE)

  if (ncol(glne_df) >= 2) {
    first_col <- normalize_barcode(glne_df[[1]])
    overlap_first_col <- sum(first_col %in% expected_cells)

    if (overlap_first_col > 10) {
      rownames(glne_df) <- first_col
      glne_df <- glne_df[, -1, drop = FALSE]
    }
  }

  glne_mat <- as.matrix(glne_df)
  suppressWarnings(storage.mode(glne_mat) <- "numeric")

  if (!is.null(rownames(glne_obj))) {
    if (is.null(rownames(glne_mat)) || any(rownames(glne_mat) == "")) {
      rownames(glne_mat) <- normalize_barcode(rownames(glne_obj))
    }
  }

  if (!is.null(rownames(glne_mat))) {
    rownames(glne_mat) <- normalize_barcode(rownames(glne_mat))
  }

  colnames(glne_mat) <- as.character(colnames(glne_mat))

  row_overlap_cells <- sum(rownames(glne_mat) %in% expected_cells)
  col_overlap_cells <- sum(normalize_barcode(colnames(glne_mat)) %in% expected_cells)

  row_overlap_genes <- sum(rownames(glne_mat) %in% expected_genes)
  col_overlap_genes <- sum(colnames(glne_mat) %in% expected_genes)

  cat("[INFO] GLNE row overlap with cells:", row_overlap_cells, "\n")
  cat("[INFO] GLNE col overlap with cells:", col_overlap_cells, "\n")
  cat("[INFO] GLNE row overlap with genes:", row_overlap_genes, "\n")
  cat("[INFO] GLNE col overlap with genes:", col_overlap_genes, "\n")

  if (
    col_overlap_cells > row_overlap_cells ||
    row_overlap_genes > col_overlap_genes ||
    (nrow(glne_mat) == length(expected_genes) && ncol(glne_mat) == length(expected_cells))
  ) {
    cat("[WARNING] GLNE seems gene x cell. Transposing to cell x gene...\n")
    glne_mat <- t(glne_mat)
    rownames(glne_mat) <- normalize_barcode(rownames(glne_mat))
  }

  if (
    (is.null(rownames(glne_mat)) || all(rownames(glne_mat) == "")) &&
    nrow(glne_mat) == length(expected_cells)
  ) {
    rownames(glne_mat) <- expected_cells
  }

  if (
    (is.null(colnames(glne_mat)) || all(colnames(glne_mat) == "")) &&
    ncol(glne_mat) == length(expected_genes)
  ) {
    colnames(glne_mat) <- expected_genes
  }

  if (sum(colnames(glne_mat) %in% expected_genes) < 10 && ncol(glne_mat) == length(expected_genes)) {
    cat("[WARNING] GLNE column names do not match HVGs, but column number matches. Assign HVG names.\n")
    colnames(glne_mat) <- expected_genes
  }

  common_cells <- expected_cells[expected_cells %in% rownames(glne_mat)]

  if (length(common_cells) == 0) {
    if (nrow(glne_mat) == length(expected_cells)) {
      cat("[WARNING] GLNE rownames do not match cells, but row number matches. Assign expected cell names.\n")
      rownames(glne_mat) <- expected_cells
      common_cells <- expected_cells
    } else {
      stop("[ERROR] No common cells between expression and GLNE.")
    }
  }

  common_genes <- expected_genes[expected_genes %in% colnames(glne_mat)]

  if (length(common_genes) == 0) {
    if (ncol(glne_mat) == length(expected_genes)) {
      cat("[WARNING] GLNE colnames do not match genes, but column number matches. Assign expected gene names.\n")
      colnames(glne_mat) <- expected_genes
      common_genes <- expected_genes
    } else {
      stop("[ERROR] No common genes between expression and GLNE.")
    }
  }

  glne_mat <- glne_mat[common_cells, common_genes, drop = FALSE]

  return(glne_mat)
}


# =========================================================
# 3. Read raw expression matrix
# =========================================================

cat("[INFO] Loading matrix from:\n")
cat(MTX_PATH, "\n")

counts <- readMM(MTX_PATH)
counts <- as(counts, "dgCMatrix")

cat("[INFO] Loading genes from:\n")
cat(GENE_PATH, "\n")

genes_df <- fread(
  GENE_PATH,
  header = FALSE,
  data.table = FALSE
)

cat("[INFO] Loading barcodes from:\n")
cat(BARCODE_PATH, "\n")

barcodes_df <- fread(
  BARCODE_PATH,
  header = FALSE,
  data.table = FALSE
)

if (ncol(genes_df) >= 2) {
  genes <- genes_df[[2]]
} else {
  genes <- genes_df[[1]]
}

genes <- make_unique(genes)
barcodes <- normalize_barcode(barcodes_df[[1]])

rownames(counts) <- genes
colnames(counts) <- barcodes

cat("[INFO] Raw matrix genes x barcodes:\n")
print(dim(counts))

cat("[INFO] Number of genes:", length(genes), "\n")
cat("[INFO] Number of barcodes:", length(barcodes), "\n")


# =========================================================
# 4. Read Barcode.csv annotation
# =========================================================

cat("[INFO] Loading annotation from:\n")
cat(ANNOTATION_PATH, "\n")

annotation_df <- read_annotation_file(ANNOTATION_PATH)

cat("[INFO] Annotation after cleaning:\n")
print(dim(annotation_df))

cat("[INFO] Annotation label counts:\n")
print(table(annotation_df$Label))

if (nrow(annotation_df) != EXPECTED_CELL_NUMBER) {
  warning(
    paste0(
      "[WARNING] Annotation cell number is ", nrow(annotation_df),
      ", not expected ", EXPECTED_CELL_NUMBER, "."
    )
  )
}


# =========================================================
# 5. Align matrix and annotation
# =========================================================

cat("[INFO] Aligning expression matrix with annotated cells...\n")

matrix_barcodes <- colnames(counts)
annot_cells <- annotation_df$Cell

common_cells <- annot_cells[annot_cells %in% matrix_barcodes]


if (length(common_cells) == 0) {
  matrix_barcodes_add1 <- paste0(matrix_barcodes, "-1")
  overlap_add1 <- sum(annot_cells %in% matrix_barcodes_add1)

  if (overlap_add1 > 0) {
    cat("[WARNING] Matrix barcodes seem to lack '-1'. Add '-1' suffix to matrix barcodes.\n")
    colnames(counts) <- matrix_barcodes_add1
    matrix_barcodes <- colnames(counts)
    common_cells <- annot_cells[annot_cells %in% matrix_barcodes]
  }
}

cat("[INFO] Common annotated cells in expression matrix:\n")
print(length(common_cells))

if (length(common_cells) == 0) {
  stop("[ERROR] No overlap between Barcode.csv and expression matrix barcodes.")
}

missing_cells <- setdiff(annot_cells, common_cells)

if (length(missing_cells) > 0) {
  warning(
    paste0(
      "[WARNING] ", length(missing_cells),
      " annotated cells are not found in expression matrix."
    )
  )
}


counts_annot <- counts[, common_cells, drop = FALSE]

annotation_ordered <- annotation_df[
  match(common_cells, annotation_df$Cell),
  ,
  drop = FALSE
]

stopifnot(all(annotation_ordered$Cell == colnames(counts_annot)))

cat("[INFO] Matrix after annotation filtering, genes x cells:\n")
print(dim(counts_annot))

rm(counts)
gc()


# =========================================================
# 6. Create Seurat object from annotated cells
# =========================================================

cat("[INFO] Creating Seurat object from annotated cells...\n")

seu <- CreateSeuratObject(
  counts = counts_annot,
  project = "GSE158490_HSPC_annotated"
)

seu$Label <- annotation_ordered$Label

seu[["percent.mito"]] <- PercentageFeatureSet(
  seu,
  pattern = "^MT-"
)

qc_df <- data.frame(
  Cell = colnames(seu),
  Label = seu$Label,
  nCount_RNA = seu$nCount_RNA,
  nFeature_RNA = seu$nFeature_RNA,
  percent.mito = seu$percent.mito,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

write.csv(
  qc_df,
  OUT_QC,
  row.names = FALSE,
  quote = FALSE
)

cat("[INFO] Seurat object:\n")
print(seu)

cat("[INFO] QC summary of annotated cells:\n")
print(summary(qc_df[, c("nCount_RNA", "nFeature_RNA", "percent.mito")]))

cat("[INFO] Label counts in Seurat object:\n")
print(table(seu$Label))


# =========================================================
# 7. NormalizeData scale.factor = 10000
# =========================================================

cat("[INFO] Running NormalizeData with scale.factor = 10000...\n")

DefaultAssay(seu) <- "RNA"

seu <- NormalizeData(
  object = seu,
  normalization.method = "LogNormalize",
  scale.factor = SCALE_FACTOR,
  verbose = TRUE
)


# =========================================================
# 8. Select 2000 HVGs
# =========================================================

cat("[INFO] Selecting HVGs...\n")

seu <- FindVariableFeatures(
  object = seu,
  selection.method = "vst",
  nfeatures = N_HVG,
  verbose = TRUE
)

hvg_genes <- VariableFeatures(seu)

cat("[INFO] Number of HVGs:\n")
print(length(hvg_genes))

writeLines(hvg_genes, OUT_HVG)


# =========================================================
# 9. Extract normalized HVG expression without MinMax
#    This matrix is used for GLNE calculation.
# =========================================================

cat("[INFO] Extracting normalized HVG expression without MinMax...\n")

norm_data <- get_assay_data_v5(
  obj = seu,
  assay = "RNA",
  layer = "data"
)

expr_norm_hvg_gene_by_cell <- norm_data[
  hvg_genes,
  colnames(seu),
  drop = FALSE
]

expr_norm_hvg_cell_by_gene <- t(as.matrix(expr_norm_hvg_gene_by_cell))

cat("[INFO] Expression matrix for GLNE, cell x gene:\n")
print(dim(expr_norm_hvg_cell_by_gene))

write_matrix_csv(
  expr_norm_hvg_cell_by_gene,
  OUT_EXPR_NORM_NO_MINMAX
)

cat("[OK] Normalized HVG expression without MinMax saved:\n")
cat(OUT_EXPR_NORM_NO_MINMAX, "\n")


# =========================================================
# 10. Save labels
# =========================================================

label_df <- data.frame(
  Cell = colnames(seu),
  Label = as.character(seu$Label),
  stringsAsFactors = FALSE,
  check.names = FALSE
)

write.csv(
  label_df,
  OUT_LABEL,
  row.names = FALSE,
  quote = FALSE
)

label_counts <- as.data.frame(table(label_df$Label))
colnames(label_counts) <- c("Label", "Count")

write.csv(
  label_counts,
  OUT_LABEL_COUNTS,
  row.names = FALSE,
  quote = FALSE
)

cat("[OK] Labels saved:\n")
cat(OUT_LABEL, "\n")

cat("[INFO] Final label counts:\n")
print(label_counts)


# =========================================================
# 11. Calculate GLNE using CellEnergy
# =========================================================

cat("[INFO] Running CellEnergy::calcGEn...\n")
cat("[INFO] GLNE input file:\n")
cat(OUT_EXPR_NORM_NO_MINMAX, "\n")

cellenergy_result <- run_calcGEn_compat(
  OUT_EXPR_NORM_NO_MINMAX
)

saveRDS(cellenergy_result, OUT_CELLENERGY_RDS)

cat("[OK] CellEnergy result saved:\n")
cat(OUT_CELLENERGY_RDS, "\n")

glne_obj <- find_glne_from_result(cellenergy_result)

cat("[INFO] Raw GLNE object dimension:\n")
print(dim(glne_obj))


# =========================================================
# 12. Fix GLNE orientation and align with expression
# =========================================================

cat("[INFO] Fixing GLNE orientation and aligning cells/genes...\n")

expected_cells <- rownames(expr_norm_hvg_cell_by_gene)
expected_genes <- colnames(expr_norm_hvg_cell_by_gene)

glne_cell_by_gene <- fix_glne_orientation(
  glne_obj = glne_obj,
  expected_cells = expected_cells,
  expected_genes = expected_genes
)

cat("[INFO] GLNE matrix after alignment, cell x gene:\n")
print(dim(glne_cell_by_gene))

write_matrix_csv(
  glne_cell_by_gene,
  OUT_GLNE_RAW
)

cat("[OK] Raw GLNE saved:\n")
cat(OUT_GLNE_RAW, "\n")


# =========================================================
# 13. MinMax scaling after GLNE calculation
# =========================================================

cat("[INFO] Applying MinMax scaling after GLNE calculation...\n")

expr_minmax <- minmax_scale_by_column(
  expr_norm_hvg_cell_by_gene
)

glne_minmax <- minmax_scale_by_column(
  glne_cell_by_gene
)

common_cells_final <- intersect(rownames(expr_minmax), rownames(glne_minmax))
common_cells_final <- rownames(expr_norm_hvg_cell_by_gene)[
  rownames(expr_norm_hvg_cell_by_gene) %in% common_cells_final
]

common_genes_final <- intersect(colnames(expr_minmax), colnames(glne_minmax))
common_genes_final <- hvg_genes[hvg_genes %in% common_genes_final]

expr_minmax <- expr_minmax[
  common_cells_final,
  common_genes_final,
  drop = FALSE
]

glne_minmax <- glne_minmax[
  common_cells_final,
  common_genes_final,
  drop = FALSE
]

label_final <- label_df[
  match(common_cells_final, label_df$Cell),
  ,
  drop = FALSE
]

rownames(label_final) <- NULL

cat("[INFO] Final View 1 dimension:\n")
print(dim(expr_minmax))

cat("[INFO] Final View 2 dimension:\n")
print(dim(glne_minmax))

cat("[INFO] Final label dimension:\n")
print(dim(label_final))

cat("[INFO] Final label counts:\n")
print(table(label_final$Label))


# =========================================================
# 14. Save final model-ready files
# =========================================================

write_matrix_csv(
  expr_minmax,
  OUT_VIEW1_MINMAX
)

write_matrix_csv(
  expr_minmax,
  OUT_VIEW1_COMPAT
)

write_matrix_csv(
  glne_minmax,
  OUT_VIEW2_MINMAX
)

write.csv(
  label_final,
  OUT_LABEL,
  row.names = FALSE,
  quote = FALSE
)

saveRDS(seu, OUT_SEURAT_RDS)

cat("[OK] Final View 1 saved:\n")
cat(OUT_VIEW1_MINMAX, "\n")

cat("[OK] Compatibility View 1 saved:\n")
cat(OUT_VIEW1_COMPAT, "\n")

cat("[OK] Final View 2 GLNE saved:\n")
cat(OUT_VIEW2_MINMAX, "\n")

cat("[OK] Final labels saved:\n")
cat(OUT_LABEL, "\n")

cat("[OK] Final Seurat object saved:\n")
cat(OUT_SEURAT_RDS, "\n")


# =========================================================
# 15. Final report
# =========================================================

sink(OUT_REPORT)

cat("GSE158490 Barcode.csv annotation preprocessing report\n")
cat("=====================================================\n\n")

cat("Workflow\n")
cat("------------------------------------\n")
cat("Use Barcode.csv as author celltype annotation\n")
cat("No PCA\n")
cat("No UMAP\n")
cat("No clustering\n")
cat("Annotated cells + HVG + GLNE + MinMax\n\n")

cat("Input files\n")
cat("------------------------------------\n")
cat("MTX_PATH:", MTX_PATH, "\n")
cat("GENE_PATH:", GENE_PATH, "\n")
cat("BARCODE_PATH:", BARCODE_PATH, "\n")
cat("ANNOTATION_PATH:", ANNOTATION_PATH, "\n\n")

cat("Raw matrix\n")
cat("------------------------------------\n")
cat("Raw genes x barcodes:", dim(counts_annot)[1], "x", length(barcodes), "\n\n")

cat("Annotated matrix\n")
cat("------------------------------------\n")
cat("Genes x annotated cells:", dim(counts_annot)[1], "x", dim(counts_annot)[2], "\n\n")

cat("Final model inputs\n")
cat("------------------------------------\n")
cat("View 1 expression MinMax:", dim(expr_minmax)[1], "x", dim(expr_minmax)[2], "\n")
cat("View 2 GLNE MinMax:", dim(glne_minmax)[1], "x", dim(glne_minmax)[2], "\n")
cat("Labels:", dim(label_final)[1], "x", dim(label_final)[2], "\n\n")

cat("Label counts\n")
cat("------------------------------------\n")
print(table(label_final$Label))
cat("\n\n")

cat("QC summary of annotated cells\n")
cat("------------------------------------\n")
print(summary(qc_df[, c("nCount_RNA", "nFeature_RNA", "percent.mito")]))
cat("\n\n")

cat("Output files\n")
cat("------------------------------------\n")
cat("Normalized HVG expression without MinMax:\n")
cat(OUT_EXPR_NORM_NO_MINMAX, "\n\n")

cat("Raw GLNE:\n")
cat(OUT_GLNE_RAW, "\n\n")

cat("View 1 expression MinMax:\n")
cat(OUT_VIEW1_MINMAX, "\n\n")

cat("View 1 compatibility file:\n")
cat(OUT_VIEW1_COMPAT, "\n\n")

cat("View 2 GLNE MinMax:\n")
cat(OUT_VIEW2_MINMAX, "\n\n")

cat("Labels:\n")
cat(OUT_LABEL, "\n\n")

cat("Label counts:\n")
cat(OUT_LABEL_COUNTS, "\n\n")

cat("QC metrics:\n")
cat(OUT_QC, "\n\n")

cat("Seurat object:\n")
cat(OUT_SEURAT_RDS, "\n\n")

sink()

cat("[OK] Report saved:\n")
cat(OUT_REPORT, "\n")

cat("[DONE] GSE158490 Barcode.csv annotation + HVG + GLNE + MinMax finished.\n")

[INFO] Loading matrix from:
/home/zhanghang/GSE158490/GSE158490_matrix.mtx 


'as(<dgTMatrix>, "dgCMatrix")' is deprecated.
Use 'as(., "CsparseMatrix")' instead.
See help("Deprecated") and help("Matrix-deprecated").



[INFO] Loading genes from:
/home/zhanghang/GSE158490/GSE158490_genes.tsv 
[INFO] Loading barcodes from:
/home/zhanghang/GSE158490/GSE158490_barcodes.tsv 
[INFO] Raw matrix genes x barcodes:
[1]  33694 737280
[INFO] Number of genes: 33694 
[INFO] Number of barcodes: 737280 
[INFO] Loading annotation from:
/home/zhanghang/GSE158490/Barcode.csv 
[INFO] Annotation file dimension:
[1] 5183    2
[INFO] Annotation columns:
[1] "V1"       "celltype"
[WARNING] Cell column not explicitly found. Use first column as cell column:
V1 
[INFO] Annotation after cleaning:
[1] 5183    2
[INFO] Annotation label counts:

    CLP     GMP HSC/MPP     MEP     mid 
   1203     789    1408     591    1192 
[INFO] Aligning expression matrix with annotated cells...
[INFO] Common annotated cells in expression matrix:
[1] 5183
[INFO] Matrix after annotation filtering, genes x cells:
[1] 33694  5183


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,4594248,245.4,7374510,393.9,5932994,316.9
Vcells,26295138,200.7,116225663,886.8,145282076,1108.5


[INFO] Creating Seurat object from annotated cells...


...

[INFO] Selecting HVGs...


Finding variable features for layer counts



[INFO] Number of HVGs:
[1] 2000
[INFO] Extracting normalized HVG expression without MinMax...
[INFO] Assay: RNA 
[INFO] Available layers:
[1] "counts" "data"  
[INFO] Expression matrix for GLNE, cell x gene:
[1] 5183 2000
[OK] Normalized HVG expression without MinMax saved:
/home/zhanghang/GSE158490/GSE158490_Barcode_model_ready/GSE158490_norm_HVG_log1p_no_MinMax_cell_by_gene.csv 
[OK] Labels saved:
/home/zhanghang/GSE158490/GSE158490_Barcode_model_ready/final_cell_labels.csv 
[INFO] Final label counts:
    Label Count
1     CLP  1203
2     GMP   789
3 HSC/MPP  1408
4     MEP   591
5     mid  1192
[INFO] Running CellEnergy::calcGEn...
[INFO] GLNE input file:
/home/zhanghang/GSE158490/GSE158490_Barcode_model_ready/GSE158490_norm_HVG_log1p_no_MinMax_cell_by_gene.csv 


2026-06-12 00:29:48.84173: Starting calculation.

2026-06-12 00:34:08.415799: Complete GLNE and cell Energy calculation.



[OK] CellEnergy result saved:
/home/zhanghang/GSE158490/GSE158490_Barcode_model_ready/CellEnergy_calcGEn_result.rds 
[INFO] Raw GLNE object dimension:
[1] 5183 2000
[INFO] Fixing GLNE orientation and aligning cells/genes...
[INFO] GLNE row overlap with cells: 5183 
[INFO] GLNE col overlap with cells: 0 
[INFO] GLNE row overlap with genes: 0 
[INFO] GLNE col overlap with genes: 2000 
[INFO] GLNE matrix after alignment, cell x gene:
[1] 5183 2000
[OK] Raw GLNE saved:
/home/zhanghang/GSE158490/GSE158490_Barcode_model_ready/GSE158490_GLNE_raw_cell_by_gene.csv 
[INFO] Applying MinMax scaling after GLNE calculation...
[INFO] Final View 1 dimension:
[1] 5183 2000
[INFO] Final View 2 dimension:
[1] 5183 2000
[INFO] Final label dimension:
[1] 5183    2
[INFO] Final label counts:

    CLP     GMP HSC/MPP     MEP     mid 
   1203     789    1408     591    1192 
[OK] Final View 1 saved:
/home/zhanghang/GSE158490/GSE158490_Barcode_model_ready/final_norm_HVG_log1p_MinMax_cell_by_gene.csv 
[OK] Comp